# Plan maestro de producción

## Tipo de problema

Determinar cuánto fabricar de cada producto cada mes balanceando costes de producción, contratación, despidos, horas extra y almacenamiento.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Programación lineal | Planificar cantidades divisibles con costes lineales. | Alternativa |
| Entera o mixta | Incluir lotes enteros, plantilla y costes fijos. | Alternativa |
| Programación dinámica | Resolver y reutilizar el coste mínimo de cada estado (mes, stock). | **Principal** |

**Enfoque utilizado:** Programación dinámica.

**Técnica concreta:** Estados (mes, stock), comparando producción normal y extra.

## Ejercicio propuesto

Plan de producción: entregar 3, 5 y 2 unidades en tres meses.  
Cada mes permite hasta 4 unidades normales (10 €/ud) y 2 extra (16 €/ud).  
Almacenar cuesta 2 €/ud/mes. Stock inicial y final cero; no hay faltantes.  
Objetivo: minimizar producción más inventario, con plantilla fija.  
Método exacto: programación dinámica con estado (mes, stock).

## Método elegido para este ejercicio

**Enfoque utilizado:** Programación dinámica.

Programación dinámica: cada estado indica el mes y el stock disponible. Se comparan las cantidades normales y extra permitidas, y se guarda el coste mínimo restante. El ejemplo fija la plantilla y simplifica el problema general.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from functools import lru_cache


def resolver():
    demanda = (3, 5, 2)

    @lru_cache(None)
    def coste(mes, stock):
        if mes == len(demanda):
            return (0, ()) if stock == 0 else (float('inf'), ())
        mejor = (float('inf'), ())
        for normal in range(5):
            for extra in range(3):
                final = stock + normal + extra - demanda[mes]
                if final < 0 or final > sum(demanda[mes + 1:]):
                    continue
                futuro, plan = coste(mes + 1, final)
                total = 10 * normal + 16 * extra + 2 * final + futuro
                if total < mejor[0]:
                    mejor = total, ((normal, extra, final),) + plan
        return mejor

    total, plan = coste(0, 0)
    assert len(plan) == len(demanda)
    stock = 0
    for entrega, (normal, extra, final) in zip(demanda, plan):
        assert 0 <= normal <= 4 and 0 <= extra <= 2
        assert stock + normal + extra - entrega == final >= 0
        stock = final
    assert stock == 0
    return total, plan

## Resolución y resultados

In [2]:
total, plan = resolver()
for mes, (normal, extra, stock) in enumerate(plan, 1):
    print(f'Mes {mes}: normal={normal}, extra={extra}, stock final={stock}')
print(f'Coste mínimo: {total:.2f} €')

Mes 1: normal=4, extra=0, stock final=1
Mes 2: normal=4, extra=0, stock final=0
Mes 3: normal=2, extra=0, stock final=0
Coste mínimo: 102.00 €


## Interpretación

Fabricar 4, 4 y 2 unidades normales, sin horas extra. Guardar 1 unidad al terminar el primer mes permite cubrir el segundo. El coste mínimo es **102 €**: 100 € de producción y 2 € de almacenamiento.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.